In [1]:
#undef __noinline__

# CUDA Fundamentals

This notebook covers the core concepts that sit underneath everything else in CUDA. You have already written kernels, allocated memory, and launched threads — now we are going to understand *why* those things work the way they do.

We will cover:
- The GPU memory hierarchy and what each type of memory actually is
- Why memory access patterns matter enormously for performance (coalescing)
- What a warp is and how the GPU actually schedules work
- Thread divergence and why it costs you performance
- Occupancy — how full your GPU actually is when a kernel runs
- Error checking, which you should be doing on every CUDA call

## 1. The Memory Hierarchy

The GPU has several distinct types of memory, each with different speed and scope. Getting this wrong is the single most common reason CUDA code is slow.

| Memory Type | Location | Speed | Scope | Lifetime |
|---|---|---|---|---|
| **Registers** | On-chip | Fastest | Single thread | Thread |
| **Shared** | On-chip | Very fast | All threads in a block | Block |
| **Local** | Off-chip (DRAM) | Slow | Single thread | Thread |
| **Global** | Off-chip (DRAM) | Slow | All threads, all blocks | Application |
| **Constant** | Off-chip, cached | Fast for reads | All threads, read-only | Application |

Everything you have been calling `cudaMalloc` for lives in **global memory** — the large, slow VRAM on the card. Shared memory is the small, fast scratchpad that threads in the same block can use to cooperate, which is what you saw in the dot product notebook.

**Local memory** sounds like it would be fast, but it is not. It is just the name given to register spill — when a thread runs out of registers, the compiler moves variables to DRAM and calls it local memory. You generally want to avoid it.

### Constant Memory

The new syntax here is `__constant__`. It declares a variable in constant memory — a small (64KB) read-only cache that is very fast when all threads in a warp read the same address simultaneously.

Here we write a value into constant memory from the host and then read it from a kernel. Notice that threads never pass it as a parameter — they just reference the global symbol directly.

In [2]:
#include <cstdio>
#include <iostream>


__constant__ float d_scale;

__global__ void set_constant(float val) {
    d_scale = val;
}

__global__ void scale_kernel(float* output, int n) {
    int tid = threadIdx.x;
    if (tid < n) {
        output[tid] = (float)tid * d_scale;
    }
}

In [3]:
void run_example() {
    const int N = 8;
    float h_out[N];
    float* d_out;
    float val_to_set = 2.5f;

    cudaMalloc(&d_out, N * sizeof(float));

    // INSTEAD of cudaMemcpyToSymbol (which is failing),
    // we launch a 1-thread kernel to set the value.
    set_constant<<<1, 1>>>(val_to_set);
    cudaDeviceSynchronize(); 

    // Now d_scale is set! Launch the work kernel.
    scale_kernel<<<1, N>>>(d_out, N);
    cudaDeviceSynchronize();

    cudaMemcpy(h_out, d_out, N * sizeof(float), cudaMemcpyDeviceToHost);

    for (int i = 0; i < N; i++) {
        printf("Index %d: %.2f\n", i, h_out[i]);
    }

    cudaFree(d_out);
}

run_example();

Index 0: 0.00
Index 1: 2.50
Index 2: 5.00
Index 3: 7.50
Index 4: 10.00
Index 5: 12.50
Index 6: 15.00
Index 7: 17.50


## 2. Memory Coalescing

Global memory is slow, but *how* you access it matters just as much as *how often* you access it. The GPU does not load individual bytes — it loads memory in 128-byte transactions. If threads in a warp access consecutive addresses, they all get served by a single transaction. If they access scattered addresses, each one may require its own transaction.

This is called **coalescing**.

**Coalesced (fast):** Thread 0 reads index 0, Thread 1 reads index 1, Thread 2 reads index 2 ... all 32 threads get their data in one transaction.

**Uncoalesced (slow):** Thread 0 reads index 0, Thread 1 reads index 32, Thread 2 reads index 64 ... each thread triggers its own transaction. 32x more memory bandwidth used.

The rule is simple: **threads with adjacent IDs should access adjacent memory addresses.**

In [4]:
#define SIZE (1 << 22)

// COALESCED: thread i reads element i — adjacent threads, adjacent addresses.
__global__ void coalesced_read(float* in, float* out, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n)
        out[tid] = in[tid] * 2.0f;
}

// UNCOALESCED: thread i reads element i * 32 — adjacent threads, scattered addresses.
__global__ void uncoalesced_read(float* in, float* out, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    int scattered = (tid * 32) % n;
    if (tid < n)
        out[tid] = in[scattered] * 2.0f;
}

In [5]:
float *d_in, *d_out;
cudaMalloc(&d_in,  SIZE * sizeof(float));
cudaMalloc(&d_out, SIZE * sizeof(float));
cudaMemset(d_in, 0x3f, SIZE * sizeof(float));

int threads = 256;
int blocks  = (SIZE + threads - 1) / threads;

cudaEvent_t start, stop;
cudaEventCreate(&start);
cudaEventCreate(&stop);
float t_coalesced, t_uncoalesced;

// Warm up
coalesced_read<<<blocks, threads>>>(d_in, d_out, SIZE);
cudaDeviceSynchronize();

cudaEventRecord(start);
coalesced_read<<<blocks, threads>>>(d_in, d_out, SIZE);
cudaEventRecord(stop);
cudaEventSynchronize(stop);
cudaEventElapsedTime(&t_coalesced, start, stop);

cudaEventRecord(start);
uncoalesced_read<<<blocks, threads>>>(d_in, d_out, SIZE);
cudaEventRecord(stop);
cudaEventSynchronize(stop);
cudaEventElapsedTime(&t_uncoalesced, start, stop);

printf("Coalesced:   %.3f ms\n", t_coalesced);
printf("Uncoalesced: %.3f ms\n", t_uncoalesced);
printf("Slowdown: %.2fx\n", t_uncoalesced / t_coalesced);

cudaEventDestroy(start);
cudaEventDestroy(stop);
cudaFree(d_in);
cudaFree(d_out);

Coalesced:   0.422 ms
Uncoalesced: 2.377 ms
Slowdown: 5.64x


## 3. Warps

You launch threads in blocks, but the GPU does not actually execute threads one at a time, or even one block at a time. The hardware groups 32 threads together into a unit called a **warp**, and all 32 threads in a warp execute the same instruction in lockstep.

This is SIMT — Single Instruction, Multiple Threads.

Key facts:
- A warp is always **32 threads**
- Threads 0–31 form warp 0, threads 32–63 form warp 1, and so on
- The SM (Streaming Multiprocessor) is the physical unit that runs warps
- When one warp is waiting on memory, the SM switches to another warp — this is how the GPU hides memory latency

The warp is why your block sizes should always be **multiples of 32**. If you launch 33 threads in a block, you get 2 warps, but the second warp has 31 idle threads — wasted hardware.

In [6]:
// This kernel identifies which warp each thread belongs to.
// threadIdx.x / 32 gives the warp index within a block.
__global__ void warp_info() {
    int tid     = threadIdx.x;
    int warp_id = tid / 32;
    int lane    = tid % 32; // position within the warp (0-31)
    
    // Only the first lane of each warp prints
    if (lane == 0)
        printf("  Warp %d starts at thread %d\n", warp_id, tid);
}

for (int bs : {33, 32, 256}) {
    int warps  = (bs + 31) / 32;
    int wasted = warps * 32 - bs;
    printf("Block size %d: %d warp%s (%d idle threads in last warp)\n",
           bs, warps, warps == 1 ? " " : "s", wasted);
}

Block size 33: 2 warps (31 idle threads in last warp)
Block size 32: 1 warp  (0 idle threads in last warp)
Block size 256: 8 warps (0 idle threads in last warp)


## 4. Thread Divergence

Because all 32 threads in a warp execute the same instruction, **branching within a warp is expensive**.

If an `if` statement causes some threads in a warp to take one path and others to take a different path, the GPU serializes them. It runs the first path with some threads active and the rest masked off, then runs the second path in reverse. Both paths take time even though each thread only needs one.

This is called **warp divergence**.

```
// BAD: threads 0-15 go one way, threads 16-31 go another.
// The entire warp runs BOTH branches, serialized.
if (threadIdx.x < 16) {
    do_path_A(); // runs with threads 16-31 masked off
} else {
    do_path_B(); // runs with threads 0-15 masked off
}

// FINE: entire warps of 32 agree on which path to take
if (blockIdx.x % 2 == 0) {
    do_path_A();
} else {
    do_path_B();
}
```

The key question to ask is: **do threads within the same warp ever disagree on which branch to take?**

In [7]:
#define DIV_SIZE (1 << 22)

// NO DIVERGENCE: branch is on blockIdx, so entire blocks (and warps) agree
__global__ void no_divergence(float* data, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= n) return;
    if (blockIdx.x % 2 == 0) {
        for(int i=0; i<200; i++) data[tid] = data[tid] * 1.0001f + 0.1f;
    } else {
        for(int i=0; i<200; i++) data[tid] = data[tid] * 0.9999f - 0.1f;
    }
}

// WITH DIVERGENCE: branch is on threadIdx, so within each warp,
// odd and even threads go different ways
__global__ void with_divergence(float* data, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid >= n) return;
    if (threadIdx.x % 2 == 0) {
        for(int i=0; i<200; i++) data[tid] = data[tid] * 1.0001f + 0.1f;
    } else {
        for(int i=0; i<200; i++) data[tid] = data[tid] * 0.9999f - 0.1f;
}
}


In [8]:
float* d_data;
cudaMalloc(&d_data, DIV_SIZE * sizeof(float));
cudaMemset(d_data, 0x3f, DIV_SIZE * sizeof(float));

int t2 = 256;
int b2 = (DIV_SIZE + t2 - 1) / t2;

cudaEvent_t s2, e2;
cudaEventCreate(&s2);
cudaEventCreate(&e2);
float t_nodiv, t_div;

// 1. Warm-up (important!)
no_divergence<<<b2, t2>>>(d_data, DIV_SIZE);
cudaDeviceSynchronize();

// 2. Time No Divergence
cudaEventRecord(s2);
no_divergence<<<b2, t2>>>(d_data, DIV_SIZE);
cudaEventRecord(e2);
cudaEventSynchronize(e2); // Force the CPU to wait here!
cudaEventElapsedTime(&t_nodiv, s2, e2);

// 3. Time With Divergence
cudaEventRecord(s2);
with_divergence<<<b2, t2>>>(d_data, DIV_SIZE);
cudaEventRecord(e2);
cudaEventSynchronize(e2); // Force the CPU to wait here!
cudaEventElapsedTime(&t_div, s2, e2);

// 4. Force the compiler to keep the math
float check;
cudaMemcpy(&check, d_data, sizeof(float), cudaMemcpyDeviceToHost);
printf("Check value (prevents optimization): %f\n", check);

printf("No divergence:   %.3f ms\n", t_nodiv);
printf("With divergence: %.3f ms\n", t_div);
printf("Overhead: %.2fx\n", t_div / t_nodiv);

Check value (prevents optimization): 62.626938
No divergence:   24.817 ms
With divergence: 59.064 ms
Overhead: 2.38x


## 5. Occupancy

**Occupancy** is the ratio of active warps on an SM to the maximum number of warps that SM can hold. Higher occupancy means more warps available to hide memory latency.

Three things limit how many warps an SM can run simultaneously:
1. **Registers** — each thread uses some number of registers. More registers per thread means fewer warps fit.
2. **Shared memory** — if your kernel allocates a lot of shared memory per block, fewer blocks fit per SM.
3. **Block size** — there is a maximum number of threads per SM.

You can query all of your GPU's limits at runtime with `cudaGetDeviceProperties`.

In [9]:
cudaDeviceProp props;
cudaGetDeviceProperties(&props, 0);

printf("GPU: %s\n", props.name);
printf("Streaming Multiprocessors: %d\n", props.multiProcessorCount);
printf("Max threads per SM: %d\n", props.maxThreadsPerMultiProcessor);
printf("Max warps per SM: %d\n",   props.maxThreadsPerMultiProcessor / props.warpSize);
printf("Max threads per block: %d\n", props.maxThreadsPerBlock);
printf("Max shared memory per block: %zu bytes (%zu KB)\n",
       props.sharedMemPerBlock, props.sharedMemPerBlock / 1024);
printf("Warp size: %d\n", props.warpSize);
printf("Registers per SM: %d\n", props.regsPerMultiprocessor);

GPU: NVIDIA GeForce GTX 1650
Streaming Multiprocessors: 16
Max threads per SM: 1024
Max warps per SM: 32
Max threads per block: 1024
Max shared memory per block: 49152 bytes (48 KB)
Warp size: 32
Registers per SM: 65536


## 6. Error Checking

Every CUDA function returns a `cudaError_t`. Almost nobody checks these in tutorial code, but in real code you absolutely should — a silently failing `cudaMalloc` or `cudaMemcpy` gives you garbage results with no indication of why.

In [10]:
#include <cstdio>
#include <cstdlib>

#define CUDA_CHECK(call) \
    do { \
        cudaError_t err = (call); \
        if (err != cudaSuccess) { \
            fprintf(stderr, "CUDA error at %s:%d — %s\n", \
                    __FILE__, __LINE__, cudaGetErrorString(err)); \
        } \
    } while (0)

__global__ void double_array(int* data, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n)
        data[tid] *= 2;
}

In [11]:
const int M = 8;
int h_data[M] = {0, 1, 2, 3, 4, 5, 6, 7};
int* d_data2;

CUDA_CHECK(cudaMalloc(&d_data2, M * sizeof(int)));
CUDA_CHECK(cudaMemcpy(d_data2, h_data, M * sizeof(int), cudaMemcpyHostToDevice));

double_array<<<1, M>>>(d_data2, M);

// Check for errors in the launch configuration
CUDA_CHECK(cudaGetLastError());
// Wait for kernel to finish and catch any runtime errors
CUDA_CHECK(cudaDeviceSynchronize());

CUDA_CHECK(cudaMemcpy(h_data, d_data2, M * sizeof(int), cudaMemcpyDeviceToHost));

printf("All CUDA calls succeeded.\n");
printf("result[0] = %d, result[1] = %d, result[2] = %d\n", h_data[0], h_data[1], h_data[2]);

CUDA_CHECK(cudaFree(d_data2));

All CUDA calls succeeded.
result[0] = 0, result[1] = 2, result[2] = 4


Kernel launches do not return an error directly. To catch kernel errors you call `cudaGetLastError()` after the launch, and `cudaDeviceSynchronize()` to wait for the kernel to finish before checking.

In [12]:
__global__ void bad_kernel() {}

// Launch with 2 billion threads in a block — impossible.
bad_kernel<<<1, 2000000000>>>();

cudaError_t err = cudaGetLastError();
if (err != cudaSuccess) {
    printf("Caught expected error: %s\n", cudaGetErrorString(err));
    cudaDeviceSynchronize(); // flush the error state
}

Caught expected error: invalid resource handle


## 7. Unified Memory

Normal CUDA requires you to manually allocate on the GPU with `cudaMalloc`, copy data to it, and copy results back. **Unified Memory** creates a single allocation that both the CPU and GPU can access — the driver automatically migrates pages between host and device as needed.

The trade-off: easier to write, but the automatic migration has overhead. For small datasets or quick prototypes it is very convenient.

In [13]:
__global__ void double_unified(int* data, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;
    if (tid < n)
        data[tid] *= 2;
}

In [14]:
const int K = 5;
int* managed;

// cudaMallocManaged creates memory accessible from both host and device
CUDA_CHECK(cudaMallocManaged(&managed, K * sizeof(int)));

// Write directly on the CPU — no memcpy needed
for (int i = 0; i < K; i++)
    managed[i] = i;

double_unified<<<1, K>>>(managed, K);

// Wait for GPU to finish before reading on CPU
CUDA_CHECK(cudaDeviceSynchronize());

// Read directly on the CPU — no memcpy needed
printf("No cudaMemcpy needed.\n");
for (int i = 0; i < K; i++)
    printf("data[%d] = %d\n", i, managed[i]);

CUDA_CHECK(cudaFree(managed));

No cudaMemcpy needed.
data[0] = 0
data[1] = 2
data[2] = 4
data[3] = 6
data[4] = 8


## Summary

| Concept | The one thing to remember |
|---|---|
| Memory hierarchy | Shared memory is fast, global memory is slow. Get data into shared memory early. |
| Constant memory | Fast read-only broadcast. Use for parameters that never change mid-kernel. |
| Coalescing | Adjacent thread IDs must read adjacent addresses. Scattered access kills bandwidth. |
| Warps | 32 threads execute in lockstep. Block sizes should be multiples of 32. |
| Divergence | Branches within a warp serialize. Branch on block index, not thread index when possible. |
| Occupancy | More active warps per SM = better latency hiding. Query your GPU limits with `cudaGetDeviceProperties`. |
| Error checking | Wrap every CUDA call in a check. Kernel errors need `cudaGetLastError()` + `cudaDeviceSynchronize()`. |
| Unified memory | `cudaMallocManaged` removes the memcpy burden. Convenient but has migration overhead. |